# Choose the Forecast Columns by Logic

**This is your decision.** For every column in the cleaned table, you decide whether the model may use it to forecast cancellations. You decide by reasoning about **when the hotel knows the value**, not by looking at which columns predict well. The notebook never reads a cancellation outcome, so the answer cannot steer your choice.

## The forecast moment

The brief asks for a table the revenue manager opens each morning, showing for each hotel and each of the next 14 nights how many rooms to oversell. So every forecast is made:

- **on a morning M**,
- **for one hotel and one arrival night D**, between M+1 and M+14,
- **about each booking on the book for night D that morning**: will it cancel before arrival?

A column is safe only if the hotel has its value **on morning M**, and **the value in our data is the value it had then**.

## The four leakage tests

| Test | Question | Fails when, for example |
|---|---|---|
| **T1 Not the outcome** | Is the value something other than the result itself, and not written *because of* a cancellation, no-show, check-in or check-out? | `reservation_status` |
| **T2 Known that morning** | Does the hotel have this value on morning M for a booking on the book? | A value learned only at check-in |
| **T3 Will not change later** | Is the value in our data the same as on morning M? If the source says it can be amended, accumulates, or is recalculated up to arrival or cancellation, our data holds a *later* value. | A count of changes made up to check-in |
| **T4 This booking only** | Is the value built from this booking (and its guest's earlier history) alone, not by comparing it with other rows of the file, which include bookings made after M and bookings in the proof period? | A flag that compares each row with every other row |

Answer each test `"yes"`, `"no"` or `"unsure"`. A column can be an **input** only if all four answers are `"yes"`. `"unsure"` counts as a failure: exclude the column and write in the reason what would need to be verified.

**Why T3 matters most here.** The source paper says each value was taken as of the day before arrival, using the latest entry in the booking change log. A booking that was cancelled cannot have changes after its cancellation. So an amendable column can differ between cancelled and kept bookings *because of* the cancellation, and the model would learn the outcome through the back door.

## What this notebook does not claim (project rule 7)

Our data is the hotels' final export, not a snapshot taken on morning M. A `"yes"` on T3 is a judgment from the source definitions; the export cannot prove it. A live morning forecast would need timestamped reservation snapshots. The scoring also counts each night's full final book, including bookings made after M. That is the separate full-book caveat, and no column choice can fix it.

## How to work through it

1. Run the cells in order (**Shift+Enter**).
2. For each group: read the logic notes, run the evidence cell (it shows values, never outcomes), then fill the decision cell.
3. The final cell checks your answers and writes `docs/FORECAST_COLUMNS_USER_REVIEW.md`. It will not write anything while an answer is missing or inconsistent.

**Three things are already fixed, so they have no decision:**

- `is_canceled` is the **label**: the contract defines it as the target, and project rule 3 forbids it as a predictor.
- `hotel` and `arrival_date` are the contract's **grouping keys**, and `adr` and `nights` give each night's **stay value**. The scoring uses them whatever you decide. Your decision is only whether they may *also* be model inputs.
- Your timing bins from notebook 01 appear next to each column for reference. That audit asked "fixed at booking?"; this one asks "known and unchanged on the forecast morning?". The answers can differ. If you change your mind, say why in the reason.

Full paper definitions for every source column: [COLUMN_SOURCE_REFERENCE.md](../docs/COLUMN_SOURCE_REFERENCE.md).

## Setup

Loads the cleaned table saved by notebook 05 (`data/hotels_clean.pkl`). Evidence uses train and choose rows only. No cell below reads `is_canceled`, `reservation_status` or `reservation_status_date` values.

In [ ]:
from pathlib import Path
import re
import sys

import pandas as pd
from IPython.display import display

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "contract.py").exists():
        repo_root = folder
        break
if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import contract

clean_path = repo_root / "data" / "hotels_clean.pkl"
if not clean_path.exists():
    raise FileNotFoundError("data/hotels_clean.pkl is missing. Run notebooks/05_apply_cleaning_decisions.ipynb first.")
cleaned = pd.read_pickle(clean_path)

train, choose, proof = contract.split_periods(cleaned)
train_choose = pd.concat([train, choose])

outcome_columns = ["is_canceled", "reservation_status", "reservation_status_date"]

# Your timing bins from notebook 01, for reference only.
earlier_bins = {}
for line in (repo_root / "docs" / "COLUMNS_USER_REVIEW.md").read_text(encoding="utf-8").splitlines():
    match = re.match(r"\| `(\w+)` \| (\d)", line)
    if match:
        earlier_bins[match.group(1)] = int(match.group(2))


def show_evidence(columns):
    """Type, missing count and most common values on train + choose. Outcome columns are never shown."""
    rows = []
    for column in columns:
        assert column not in outcome_columns
        values = train_choose[column]
        rows.append({
            "column": column,
            "nb01 bin": earlier_bins.get(column, "built later"),
            "type": str(values.dtype),
            "distinct": values.nunique(),
            "missing": int(values.isna().sum()),
            "most common values (rows)": ", ".join(f"{v} ({n:,})" for v, n in values.value_counts().head(4).items()),
        })
    display(pd.DataFrame(rows).set_index("column"))


my_decisions = {}
VALID_ANSWERS = {"yes", "no", "unsure"}

print(f"Cleaned table: {len(cleaned):,} rows x {cleaned.shape[1]} columns; "
      f"{cleaned.shape[1] - 1} need a decision (all except the label is_canceled).")

## Group A: the night being forecast

`hotel`, `arrival_date`, `arrival_date_year`, `arrival_date_month`, `arrival_date_week_number`, `arrival_date_day_of_month`

- **`hotel`**: the property. The paper does not describe a booking moving between hotels.
- **`arrival_date`** (built in notebook 05 from the three date parts) **and its parts**: on morning M you forecast a specific night D, so D's date is certain. The narrower question is about each booking: if a booking sits on night D that morning, is its arrival date in our data still D? If the guest later moves the booking, the export shows the *new* date, and the booking lands on a different night in our data than the one it was on that morning.
- **`arrival_date_year`**: same as above. A separate point, not leakage: train holds only 2015 and 2016, while proof is 2017, a year value the model has never seen (see the evidence below).
- **`arrival_date_week_number`** and **`arrival_date_day_of_month`**: derived from the same date.

**Questions:** Is a value that describes the *night* (its month, weekday, season) different from a value that describes the *booking*? Does an amended arrival date break T3 for these columns, or only move the booking to a different night?

In [ ]:
show_evidence(["hotel", "arrival_date", "arrival_date_year", "arrival_date_month",
               "arrival_date_week_number", "arrival_date_day_of_month"])

# Which arrival years each period contains (dates only; no outcome read).
for name, period in {"train": train, "choose": choose, "proof": proof}.items():
    print(f"{name:>6}: arrival years {sorted(int(y) for y in period['arrival_date_year'].unique())}")

In [ ]:
# tests = [T1 not the outcome, T2 known that morning, T3 will not change later, T4 this booking only]
# Answer each test with "yes", "no" or "unsure". use = "input" or "exclude". Write the reason in your own words.
my_decisions.update({
    "hotel":                     {"tests": [None, None, None, None], "use": None, "reason": ""},
    "arrival_date":              {"tests": [None, None, None, None], "use": None, "reason": ""},
    "arrival_date_year":         {"tests": [None, None, None, None], "use": None, "reason": ""},
    "arrival_date_month":        {"tests": [None, None, None, None], "use": None, "reason": ""},
    "arrival_date_week_number":  {"tests": [None, None, None, None], "use": None, "reason": ""},
    "arrival_date_day_of_month": {"tests": [None, None, None, None], "use": None, "reason": ""},
})

## Group B: the guest's earlier history

`is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled`, `flag_repeat_status_conflict`

- **`is_repeated_guest`**: the paper sets it to 1 when a customer profile was linked to the booking and that profile was created *before* this booking was. That is settled when the booking is created. **Question:** can a profile be linked to the booking later, for example at check-in, turning a 0 into a 1? The paper does not say when the link is made.
- **`previous_cancellations`**: the number of bookings by the same profile, created before this booking, that were cancelled; 0 when no profile is linked. **Question:** the definition limits *when the earlier bookings were created*, not *when they were cancelled*. If an earlier booking is cancelled after morning M, does our data count it? (Notebook 05, decision C6, already plans to test this column with and without on choose.)
- **`previous_bookings_not_canceled`**: the number of earlier bookings by the same profile that were not cancelled. **Question:** an earlier booking that has not yet arrived on morning M counts as "not cancelled" then. Could it cancel later and move into the other count?
- **`flag_repeat_status_conflict`** (built in notebook 05): `is_repeated_guest` is 0 but a prior count is above 0. Built from the three columns above only, so it inherits their answers.

In [ ]:
show_evidence(["is_repeated_guest", "previous_cancellations", "previous_bookings_not_canceled",
               "flag_repeat_status_conflict"])

In [ ]:
# tests = [T1 not the outcome, T2 known that morning, T3 will not change later, T4 this booking only]
# Answer each test with "yes", "no" or "unsure". use = "input" or "exclude". Write the reason in your own words.
my_decisions.update({
    "is_repeated_guest":              {"tests": [None, None, None, None], "use": None, "reason": ""},
    "previous_cancellations":         {"tests": [None, None, None, None], "use": None, "reason": ""},
    "previous_bookings_not_canceled": {"tests": [None, None, None, None], "use": None, "reason": ""},
    "flag_repeat_status_conflict":    {"tests": [None, None, None, None], "use": None, "reason": ""},
})

## Group C: when the booking was made

`lead_time`, `days_in_waiting_list`

- **`lead_time`**: the paper defines it as days from the date the booking was entered in the system to the arrival date. The entry date never changes, so for a booking on night D on morning M, the hotel can compute D minus the entry date. **Question:** if the arrival date was later amended, our data measures lead time to the *new* date. That ties this column to your answer on arrival dates in Group A.
- **`days_in_waiting_list`**: days from entering the system until the booking was confirmed to the customer. It is known once the booking is confirmed. **Question:** can a booking still on the waiting list on morning M be on the book you forecast? If it can, its final value is not yet known that morning.

In [ ]:
show_evidence(["lead_time", "days_in_waiting_list"])

In [ ]:
# tests = [T1 not the outcome, T2 known that morning, T3 will not change later, T4 this booking only]
# Answer each test with "yes", "no" or "unsure". use = "input" or "exclude". Write the reason in your own words.
my_decisions.update({
    "lead_time":            {"tests": [None, None, None, None], "use": None, "reason": ""},
    "days_in_waiting_list": {"tests": [None, None, None, None], "use": None, "reason": ""},
})

## Group D: stay length and party size

`stays_in_weekend_nights`, `stays_in_week_nights`, `nights`, `adults`, `children`, `babies`, `guests`, and the flags built from them: `flag_zero_nights`, `flag_zero_guests`, `flag_children_without_adults`, `flag_ten_plus_adults`, `flag_children_missing`

- **Stay nights**: the paper calculates them from the arrival and departure dates and takes the latest change-log value up to the day before arrival. The authors warn that stay length and guest details can change, including at check-in.
- **`adults`, `children`, `babies`**: from the bookings table and change log, with the same warning.
- **`nights`** (weekend + week nights) and **`guests`** (adults + children + babies) were built in notebook 05; they inherit the answers of the columns they add up.
- **The flags** were built in notebook 05 from these columns and inherit their answers. `flag_children_missing` marks rows whose `children` is blank in the export; the blank describes the export, not what the hotel saw that morning.

**Questions:** If the party size or stay length changes between morning M and arrival, which value does our data hold? For a cancelled booking, can the values have changed after the cancellation?

In [ ]:
show_evidence(["stays_in_weekend_nights", "stays_in_week_nights", "nights", "adults", "children", "babies",
               "guests", "flag_zero_nights", "flag_zero_guests", "flag_children_without_adults",
               "flag_ten_plus_adults", "flag_children_missing"])

In [ ]:
# tests = [T1 not the outcome, T2 known that morning, T3 will not change later, T4 this booking only]
# Answer each test with "yes", "no" or "unsure". use = "input" or "exclude". Write the reason in your own words.
my_decisions.update({
    "stays_in_weekend_nights":      {"tests": [None, None, None, None], "use": None, "reason": ""},
    "stays_in_week_nights":         {"tests": [None, None, None, None], "use": None, "reason": ""},
    "nights":                       {"tests": [None, None, None, None], "use": None, "reason": ""},
    "adults":                       {"tests": [None, None, None, None], "use": None, "reason": ""},
    "children":                     {"tests": [None, None, None, None], "use": None, "reason": ""},
    "babies":                       {"tests": [None, None, None, None], "use": None, "reason": ""},
    "guests":                       {"tests": [None, None, None, None], "use": None, "reason": ""},
    "flag_zero_nights":             {"tests": [None, None, None, None], "use": None, "reason": ""},
    "flag_zero_guests":             {"tests": [None, None, None, None], "use": None, "reason": ""},
    "flag_children_without_adults": {"tests": [None, None, None, None], "use": None, "reason": ""},
    "flag_ten_plus_adults":         {"tests": [None, None, None, None], "use": None, "reason": ""},
    "flag_children_missing":        {"tests": [None, None, None, None], "use": None, "reason": ""},
})

## Group E: what was booked, and the price

`meal`, `reserved_room_type`, `assigned_room_type`, `adr`, `required_car_parking_spaces`, `total_of_special_requests`, and the flags `flag_room_type_p`, `flag_adr_above_1000`, `flag_adr_negative`

- **`meal`**: from the bookings, change-log and meals tables. The paper lists meal among the attributes that can be amended. (Notebook 05 recoded `Undefined` to `SC`.)
- **`reserved_room_type`**: the room type requested; can be amended.
- **`assigned_room_type`**: the paper says it can differ from the reserved type *for hotel operation reasons, such as overbooking*, or at the customer's request. **Question:** when is a room assigned: at booking, close to arrival, or at check-in? An assignment caused by overbooking is a result of the very decision this forecast feeds.
- **`adr`**: the paper calculates it as the sum of all lodging transactions divided by the number of staying nights. **Question:** are lodging transactions posted when the booking is made, or while the guest stays? For a guest who stayed, those transactions come after morning M. For a cancelled booking, which transactions exist at all? (The contract also uses `adr` × `nights` to value an empty room. That is scoring and is separate from using `adr` as an input.)
- **`required_car_parking_spaces`** and **`total_of_special_requests`**: guest requests from the bookings table and change log; the special-requests total is a sum of all requests made, so it can grow until arrival.
- **The flags** inherit the answers of `reserved_room_type` and `adr`.

In [ ]:
show_evidence(["meal", "reserved_room_type", "assigned_room_type", "adr", "required_car_parking_spaces",
               "total_of_special_requests", "flag_room_type_p", "flag_adr_above_1000", "flag_adr_negative"])

differs = train_choose["assigned_room_type"].ne(train_choose["reserved_room_type"])
print(f"assigned_room_type differs from reserved_room_type on {differs.sum():,} train + choose bookings ({differs.mean():.1%})")

In [ ]:
# tests = [T1 not the outcome, T2 known that morning, T3 will not change later, T4 this booking only]
# Answer each test with "yes", "no" or "unsure". use = "input" or "exclude". Write the reason in your own words.
my_decisions.update({
    "meal":                        {"tests": [None, None, None, None], "use": None, "reason": ""},
    "reserved_room_type":          {"tests": [None, None, None, None], "use": None, "reason": ""},
    "assigned_room_type":          {"tests": [None, None, None, None], "use": None, "reason": ""},
    "adr":                         {"tests": [None, None, None, None], "use": None, "reason": ""},
    "required_car_parking_spaces": {"tests": [None, None, None, None], "use": None, "reason": ""},
    "total_of_special_requests":   {"tests": [None, None, None, None], "use": None, "reason": ""},
    "flag_room_type_p":            {"tests": [None, None, None, None], "use": None, "reason": ""},
    "flag_adr_above_1000":         {"tests": [None, None, None, None], "use": None, "reason": ""},
    "flag_adr_negative":           {"tests": [None, None, None, None], "use": None, "reason": ""},
})

## Group F: who sold the booking, and who the guest is

`market_segment`, `distribution_channel`, `agent`, `company`, `customer_type`, `country`

- **`market_segment`** and **`distribution_channel`**: from the bookings table, change log and their lookup tables. The paper does not say whether a booking can be reclassified later.
- **`agent`**: the travel agency that made the booking; `none` means no agency (notebook 05). **Question:** can an agency be added or changed through the change log?
- **`company`**: the company or entity that made or pays for the booking. **Question:** can the paying company be assigned later, for example at check-out?
- **`customer_type`**: Contract, Group, Transient or Transient-party. Transient-party means the booking is linked to at least one other transient booking. **Question:** can that link be made after morning M? Does it depend on other bookings (T4)?
- **`country`**: the guest's nationality. The authors warn that the correct nationality is often known only at check-in. **Question:** a guest who checked in may have had the value corrected; a cancelled booking never reached check-in. Notebook 05 labelled the missing values `unknown`.

In [ ]:
show_evidence(["market_segment", "distribution_channel", "agent", "company", "customer_type", "country"])

In [ ]:
# tests = [T1 not the outcome, T2 known that morning, T3 will not change later, T4 this booking only]
# Answer each test with "yes", "no" or "unsure". use = "input" or "exclude". Write the reason in your own words.
my_decisions.update({
    "market_segment":       {"tests": [None, None, None, None], "use": None, "reason": ""},
    "distribution_channel": {"tests": [None, None, None, None], "use": None, "reason": ""},
    "agent":                {"tests": [None, None, None, None], "use": None, "reason": ""},
    "company":              {"tests": [None, None, None, None], "use": None, "reason": ""},
    "customer_type":        {"tests": [None, None, None, None], "use": None, "reason": ""},
    "country":              {"tests": [None, None, None, None], "use": None, "reason": ""},
})

## Group G: payment and amendments

`deposit_type`, `booking_changes`

- **`deposit_type`**: the paper calculates it from the payments found in the transactions table **before the arrival or cancellation date**: no payment gives `No Deposit`, a payment at least equal to the full stay gives `Non Refund`, anything less gives `Refundable`. **Question:** a payment made after morning M still changes the value in our data. (This is also the column behind the brief's first client note, that prepaid bookings seem never to turn into stays.)
- **`booking_changes`**: the paper counts the amendments made from the moment the booking was entered **until check-in or cancellation**. **Question:** how many of those changes happen after morning M?

In [ ]:
show_evidence(["deposit_type", "booking_changes"])

In [ ]:
# tests = [T1 not the outcome, T2 known that morning, T3 will not change later, T4 this booking only]
# Answer each test with "yes", "no" or "unsure". use = "input" or "exclude". Write the reason in your own words.
my_decisions.update({
    "deposit_type":    {"tests": [None, None, None, None], "use": None, "reason": ""},
    "booking_changes": {"tests": [None, None, None, None], "use": None, "reason": ""},
})

## Group H: the outcome columns

`reservation_status`, `reservation_status_date`

- **`reservation_status`**: the booking's last status: `Canceled`, `Check-Out` or `No-Show`.
- **`reservation_status_date`**: the date that last status was set.

No evidence cell: their values are outcomes, and this notebook does not read outcomes. (`is_canceled`, the label, has no decision; see the introduction.)

In [ ]:
# tests = [T1 not the outcome, T2 known that morning, T3 will not change later, T4 this booking only]
# Answer each test with "yes", "no" or "unsure". use = "input" or "exclude". Write the reason in your own words.
my_decisions.update({
    "reservation_status":      {"tests": [None, None, None, None], "use": None, "reason": ""},
    "reservation_status_date": {"tests": [None, None, None, None], "use": None, "reason": ""},
})

## Group I: the cross-row flag

`flag_identical_group`

- Built in notebook 05 by comparing each row with **every other row in the whole file** (train, choose and proof) across all non-outcome columns, and marking rows that have an identical twin.

**Questions:** On morning M, would the hotel have the rows this flag compares against? They include bookings made after M and bookings arriving in the proof period. The comparison also uses the final values of the amendable columns in Groups C to G.

In [ ]:
show_evidence(["flag_identical_group"])

In [ ]:
# tests = [T1 not the outcome, T2 known that morning, T3 will not change later, T4 this booking only]
# Answer each test with "yes", "no" or "unsure". use = "input" or "exclude". Write the reason in your own words.
my_decisions.update({
    "flag_identical_group": {"tests": [None, None, None, None], "use": None, "reason": ""},
})

## Final cell: check your answers and save them

Run this after every decision cell is filled. It checks that:

1. every column except the label has four test answers, a `use` and a reason;
2. every **input** has four `"yes"` answers;
3. every **built** column marked as an input (from notebook 05) is built only from columns that passed all four tests themselves;
4. no outcome column is an input.

If anything fails it lists the problems and saves nothing. When everything passes it shows your input list, checks those columns for missing values (no outcome is read), and writes `docs/FORECAST_COLUMNS_USER_REVIEW.md`.

In [ ]:
TEST_NAMES = ["T1 not the outcome", "T2 known that morning", "T3 will not change later", "T4 this booking only"]
ALL_YES = ["yes"] * 4

# Columns built in notebook 05 (or from the date parts), with the source columns they are made from.
structural = [c for c in cleaned.columns[:32] if c not in outcome_columns]
built_from = {
    "arrival_date": ["arrival_date_year", "arrival_date_month", "arrival_date_day_of_month"],
    "nights": ["stays_in_weekend_nights", "stays_in_week_nights"],
    "guests": ["adults", "children", "babies"],
    "flag_identical_group": structural,
    "flag_zero_nights": ["stays_in_weekend_nights", "stays_in_week_nights"],
    "flag_zero_guests": ["adults", "children", "babies"],
    "flag_room_type_p": ["reserved_room_type"],
    "flag_children_without_adults": ["adults", "children", "babies"],
    "flag_ten_plus_adults": ["adults"],
    "flag_adr_above_1000": ["adr"],
    "flag_adr_negative": ["adr"],
    "flag_children_missing": ["children"],
    "flag_repeat_status_conflict": ["is_repeated_guest", "previous_cancellations", "previous_bookings_not_canceled"],
}

to_decide = [column for column in cleaned.columns if column != "is_canceled"]
problems = []

for column in to_decide:
    answer = my_decisions.get(column)
    if answer is None:
        problems.append(f"{column}: no decision recorded")
    elif len(answer["tests"]) != 4 or any(test not in VALID_ANSWERS for test in answer["tests"]):
        problems.append(f"{column}: answer all four tests with yes, no or unsure")
    elif answer["use"] not in {"input", "exclude"}:
        problems.append(f"{column}: set use to input or exclude")
    elif not answer["reason"].strip():
        problems.append(f"{column}: write a reason")
    elif answer["use"] == "input" and answer["tests"] != ALL_YES:
        problems.append(f"{column}: an input needs yes on all four tests")

for column in set(my_decisions) - set(to_decide):
    problems.append(f"{column}: not a column of the cleaned table")

for column, sources in built_from.items():
    if my_decisions.get(column, {}).get("use") == "input":
        failing = [s for s in sources if my_decisions.get(s, {}).get("tests") != ALL_YES]
        if failing:
            problems.append(f"{column}: built from {failing}, which did not pass all four tests")

if problems:
    print(f"Not saved. Fix these {len(problems)} item(s) first:")
    for problem in problems:
        print("  -", problem)
else:
    model_inputs = [column for column in to_decide if my_decisions[column]["use"] == "input"]

    # Hard guard (project rule 3): no outcome column can ever be an input.
    assert not set(model_inputs) & set(outcome_columns)

    report = pd.DataFrame(
        [[column, *my_decisions[column]["tests"], my_decisions[column]["use"], my_decisions[column]["reason"]]
         for column in to_decide],
        columns=["column", *TEST_NAMES, "use", "reason"],
    )
    display(report)

    # Missing values in the chosen inputs, by period (feature values only; no outcome is read).
    missing = pd.DataFrame({name: period[model_inputs].isna().sum()
                            for name, period in {"train": train, "choose": choose, "proof": proof}.items()})
    print(f"Model inputs ({len(model_inputs)}): {model_inputs}")
    print("Missing values in the inputs by period:", "none" if missing.values.sum() == 0 else "")
    if missing.values.sum():
        display(missing[missing.sum(axis=1) > 0])

    lines = [
        "# Forecast Column Decisions (User-Completed)",
        "",
        "> Decisions entered by the learner in `notebooks/06_forecast_column_selection.ipynb`, by logic only; "
        "the notebook reads no cancellation outcome. A column is an input only if it passed all four tests. "
        "These are judgments from the source definitions; the final export cannot prove the values match a "
        "morning snapshot, which would need timestamped reservation data.",
        "",
        "**Forecast moment:** morning M, for each hotel and each arrival night D from M+1 to M+14, "
        "about each booking on the book for night D.",
        "",
        "**Tests:** T1 not the outcome; T2 known that morning; T3 will not change later; T4 this booking only.",
        "",
        f"**Model inputs ({len(model_inputs)}):** " + (", ".join(f"`{c}`" for c in model_inputs) or "none"),
        "",
        "**Not decided here:** `is_canceled` is the label. `hotel` and `arrival_date` are the contract's grouping "
        "keys and `adr` and `nights` its stay-value inputs; the scoring uses them whatever is decided above.",
        "",
        "| Column | T1 | T2 | T3 | T4 | Use | Reason in my words |",
        "|---|---|---|---|---|---|---|",
    ]
    for column in to_decide:
        answer = my_decisions[column]
        reason = answer["reason"].strip().replace("|", "/")
        lines.append(f"| `{column}` | " + " | ".join(answer["tests"]) + f" | {answer['use']} | {reason} |")

    output_path = repo_root / "docs" / "FORECAST_COLUMNS_USER_REVIEW.md"
    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"Saved {output_path.relative_to(repo_root)}")